# 04 Feature engineering

**Purpose:** turn each planned stop into the inputs the Task 1 models will use, and prove that every input is something a planner actually has when the plan is drafted.

**Inputs:** `stops.parquet` from notebook 03 and the clean reference tables from notebook 02.

**Outputs:**
- `data/processed/task1_features.parquet`: one row per dispatched order (train and test) with all features and, for train, the two labels
- `reports/feature_audit.csv`: every feature, where it comes from and why it is available in time

**The one rule:** Waypoint drafts tomorrow's routes at the 4 PM cutoff. A feature is allowed only if it is known at that moment. That rules out anything measured on the delivery day itself, and even the previous day's results, because those routes may still be on the road at 4 PM.

The feature code lives in `src/xora/features.py` so that notebook 05 and the final notebook build exactly the same inputs. This notebook calls it one group at a time and shows what each group adds.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora.io import load_processed, save_processed
from xora.paths import REPORTS
from xora.plotting import apply_style, SERIES, INK
from xora import features as fe

apply_style()
pd.set_option("display.width", 160)

stops = load_processed("stops")
outlets = load_processed("outlets")
vehicles = load_processed("vehicles")
allowance = load_processed("service_allowance")
calendar = load_processed("calendar")
road = load_processed("road_conditions")
traffic = load_processed("traffic_speed")

train = stops.split == "train"
stops.groupby("split").agg(stops=("delivery_id", "size"), first_day=("date", "min"), last_day=("date", "max"))

,stops,first_day,last_day
split,,,
test,5014,2026-02-16,2026-03-28
train,91894,2024-01-01,2026-02-14


The test stops start two days after training ends, and they carry only planned times. The features below are built the same way for both, so the model never sees anything at training time that it would not have for the test.

## 2. Order and vehicle features

What is being delivered, where it goes and what it travels on. All of this is on the order and the drafted plan.

- size of the order: units, weight, volume, and weight per unit (bulky Tech items take longer to move)
- outlet: brand, district, dock type, parking constraint, whether it is a mall with a delivery slot
- vehicle: type, refrigeration and capacity
- `was_deferred`: the order was pushed from the previous day, which the planner knows
- `allowance_min`: today's published allowance for this brand and dock type, the baseline the model has to beat

In [2]:
order_f = fe.order_features(stops, outlets, vehicles, allowance)
print(order_f.shape)
order_f.head(3).T

(96908, 19)


,0,1,2
brand,Fresh,Fresh,Fresh
depot,Peliyagoda,Peliyagoda,Peliyagoda
district,Colombo,Colombo,Colombo
temp_requirement,ambient,ambient,ambient
dock_type,street,street,street
parking_constraint,van_only,van_only,van_only
vehicle_type,van,van,van
vehicle_temp,ambient,ambient,ambient
outlet_id,OUT001,OUT002,OUT003
order_units,8,15,15
